In [12]:
# Q1: Total .wav files after generate_synthetic_dataset(samples_per_genre=50)
genres = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

samples_per_genre = 50
total_files = len(genres) * samples_per_genre

print(f"Number of genres: {len(genres)}")
print(f"Samples per genre: {samples_per_genre}")
print(f"Total .wav files = {len(genres)} genres × {samples_per_genre} samples = {total_files}")

Number of genres: 10
Samples per genre: 50
Total .wav files = 10 genres × 50 samples = 500


In [13]:
import torch

target_sr = 22050
duration = 30
target_length = target_sr * duration  
channels = 2                             

print(f"target_sr    = {target_sr} Hz")
print(f"duration     = {duration} s")
print(f"target_length = {target_sr} × {duration} = {target_length} samples")
print(f"channels     = {channels}  (stems are stereo from source separation)")
print()

# Simulate the shape using a synthetic tensor of the same size
synthetic_waveform = torch.zeros(channels, target_length)
print(f"Resulting waveform tensor shape: {tuple(synthetic_waveform.shape)}")

# Answer: (2, 661500)

target_sr    = 22050 Hz
duration     = 30 s
target_length = 22050 × 30 = 661500 samples
channels     = 2  (stems are stereo from source separation)

Resulting waveform tensor shape: (2, 661500)


In [14]:
import torch
import torchaudio

target_sr = 22050
n_fft     = 2048
hop_length = 512
n_mels    = 128
N         = 661500   # samples in waveform (22050 Hz * 30 s)

# Frame count formula (torchaudio default center=True): n_frames = floor(N / hop_length) + 1
n_frames = N // hop_length + 1
print(f"N = {N} samples,  hop_length = {hop_length}")
print(f"n_frames = floor({N} / {hop_length}) + 1 = {N // hop_length} + 1 = {n_frames}")

mel_transform   = torchaudio.transforms.MelSpectrogram(
    sample_rate=target_sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

waveform = torch.zeros(2, N)          # stereo 30-second waveform
mel_spec    = mel_transform(waveform)
mel_spec_db = amplitude_to_db(mel_spec)

print(f"\nAfter MelSpectrogram:  {tuple(mel_spec.shape)}")
print(f"After AmplitudeToDB:   {tuple(mel_spec_db.shape)}")
print(f"\n=> Saved .pt tensor shape: {tuple(mel_spec_db.shape)}")

N = 661500 samples,  hop_length = 512
n_frames = floor(661500 / 512) + 1 = 1291 + 1 = 1292

After MelSpectrogram:  (2, 128, 1292)
After AmplitudeToDB:   (2, 128, 1292)

=> Saved .pt tensor shape: (2, 128, 1292)


In [15]:
import torch
import torch.nn as nn

T = 1292   # time frames from Q3
x = torch.zeros(32, 1, 128, T)
print(f"Input:{tuple(x.shape)}")

cnn_block1 = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2),
)
cnn_block2 = nn.Sequential(
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2),
)

after_block1 = cnn_block1(x)
print(f"After CNN Block1 MaxPool2d: {tuple(after_block1.shape)}")

after_block2 = cnn_block2(after_block1)
print(f"After CNN Block2 MaxPool2d: {tuple(after_block2.shape)}") 

print(f"\n=> Shape right before .permute(): {tuple(after_block2.shape)}")
print(f"   Interpretation: (Batch={after_block2.shape[0]}, "
      f"Channels={after_block2.shape[1]}, "
      f"MelBins={after_block2.shape[2]}, "
      f"TimeSteps={after_block2.shape[3]})")


Input:(32, 1, 128, 1292)
After CNN Block1 MaxPool2d: (32, 32, 64, 646)
After CNN Block2 MaxPool2d: (32, 64, 32, 323)

=> Shape right before .permute(): (32, 64, 32, 323)
   Interpretation: (Batch=32, Channels=64, MelBins=32, TimeSteps=323)


In [16]:
import torch
import torch.nn as nn


class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # TODO 1: CNN Backbone
        # Input: (Batch, 1, 128, Time)
        # Block 1: Conv2d(1->32, k=3, p=1) -> BN -> ReLU -> MaxPool2d(2)
        # Block 2: Conv2d(32->64, k=3, p=1) -> BN -> ReLU -> MaxPool2d(2)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # TODO 2: RNN
        # After CNN: (B, 64, 32, T//4) → input_size = 64 * 32 = 2048
        self.lstm = nn.LSTM(
            input_size=64 * 32,   # 2048
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        # TODO 3: Classifier
        # BiLSTM output size = hidden_size * 2 = 128
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        """
        Input:  x shape (Batch, 1, 128, Time)
        Output: logits shape (Batch, num_classes)
        """
        # TODO 4: CNN backbone
        x = self.cnn(x)              # (B, 64, 32, T//4)

        # TODO 5: Reshape for LSTM — (B, T, C*F)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)   # (B, T, C, F)
        x = x.reshape(b, t, c * f)  # (B, T, 2048)

        # TODO 6: LSTM
        x, _ = self.lstm(x)          # (B, T, 128)

        # TODO 7: Global max pooling over time
        x, _ = torch.max(x, dim=1)   # (B, 128)

        # TODO 8: Classifier
        logits = self.fc(x)           # (B, num_classes)
        return logits


# Sanity check
_model = CRNN(num_classes=10)
_total = sum(p.numel() for p in _model.parameters() if p.requires_grad)
_dummy = torch.zeros(2, 1, 128, 323)
_out   = _model(_dummy)
print(f"CRNN created. Params: {_total:,}")
print(f"Input {tuple(_dummy.shape)} → Output {tuple(_out.shape)}  ✓")
del _model, _dummy, _out


CRNN created. Params: 1,102,666
Input (2, 1, 128, 323) → Output (2, 10)  ✓


In [ ]:
import torch.nn as nn

# Q5: Trainable parameters in the LSTM layer of CRNN
# BiLSTM: input_size=2048 (= 64 channels * 32 mel-bins), hidden_size=64, bidirectional=True

model = CRNN(num_classes=10)
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

print("LSTM layer parameter breakdown:")
print(f"  For EACH direction (forward & backward):")
print(f"    W_ih (input->hidden): 4×{64} × {2048} = {4*64*2048:,}")
print(f"    W_hh (hidden->hidden):4×{64} × {64}   = {4*64*64:,}")
print(f"    b_ih:                 4×{64}           = {4*64:,}")
print(f"    b_hh:                 4×{64}           = {4*64:,}")
per_dir = 4*64*2048 + 4*64*64 + 4*64 + 4*64
print(f"    Subtotal per direction: {per_dir:,}")
print()
print(f"  Bidirectional (2 directions): {per_dir:,} × 2 = {per_dir*2:,}")
print()
print(f"Verified with PyTorch:")
print(f"  sum(p.numel() for p in model.lstm.parameters() if p.requires_grad) = {lstm_params:,}")


LSTM layer parameter breakdown:
  For EACH direction (forward & backward):
    W_ih (input->hidden): 4×64 × 2048 = 524,288
    W_hh (hidden->hidden):4×64 × 64   = 16,384
    b_ih:                 4×64           = 256
    b_hh:                 4×64           = 256
    Subtotal per direction: 541,184

  Bidirectional (2 directions): 541,184 × 2 = 1,082,368

Verified with PyTorch:
  sum(p.numel() for p in model.lstm.parameters() if p.requires_grad) = 1,082,368


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import glob, os, random, time
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── Path detection (Kaggle vs local) ──────────────────────────────────────────
KAGGLE_FEATURES = Path('/kaggle/working/features/train')
LOCAL_STEMS     = Path('/Users/sanskar/dev/DL-GenAI-P/messy_mashup/genres_stems')

GENRES = sorted(['blues', 'classical', 'country', 'disco', 'hiphop',
                 'jazz', 'metal', 'pop', 'reggae', 'rock'])
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}


class PrecomputedFeatureDataset(Dataset):
    """Loads precomputed mel-spectrogram .pt files (Kaggle mode)."""
    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        genre = Path(path).parent.name
        label = GENRE_TO_IDX[genre]
        feature = torch.load(path, weights_only=True)  
        feature = feature.mean(dim=0, keepdim=True)   
        return feature, label


class OnTheFlyDataset(Dataset):
    """Generates synthetic mashups on-the-fly from local stems (local dev)."""
    STEM_NAMES = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

    def __init__(self, stems_dir, genre_list, samples_per_genre=50,
                 target_sr=22050, duration=30, split='train', val_ratio=0.15):
        self.stems_dir  = Path(stems_dir)
        self.target_sr  = target_sr
        self.target_len = target_sr * duration
        self.mel_tf     = torchaudio.transforms.MelSpectrogram(
            sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128)
        self.db_tf      = torchaudio.transforms.AmplitudeToDB()

        all_items = []
        for genre in genre_list:
            song_dirs = sorted((self.stems_dir / genre).iterdir())
            n_val = max(1, int(len(song_dirs) * val_ratio))
            songs = song_dirs[n_val:] if split == 'train' else song_dirs[:n_val]
            for _ in range(samples_per_genre):
                all_items.append((genre, list(songs)))
        random.shuffle(all_items)
        self.items = all_items

    def _load_stem(self, song_dir):
        import soundfile as sf, numpy as np
        for sn in self.STEM_NAMES:
            p = song_dir / sn
            if not p.exists():
                continue
            data, sr = sf.read(str(p))
            if data.ndim > 1:
                data = data.mean(axis=1)
            data = data.astype(np.float32)
            if sr != self.target_sr:
                import librosa
                data = librosa.resample(data, orig_sr=sr, target_sr=self.target_sr)
            if len(data) > self.target_len:
                data = data[:self.target_len]
            elif len(data) < self.target_len:
                data = np.pad(data, (0, self.target_len - len(data)))
            return torch.tensor(data).unsqueeze(0)   # (1, N)
        return None

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        genre, songs = self.items[idx]
        chosen = random.sample(songs, min(4, len(songs)))
        stems = [s for s in (self._load_stem(p) for p in chosen) if s is not None]
        if not stems:
            return torch.zeros(1, 128, 1292), GENRE_TO_IDX[genre]
        mashup = torch.stack(stems).sum(dim=0)
        mx = mashup.abs().max()
        if mx > 0:
            mashup = mashup / mx * 0.9
        mel = self.db_tf(self.mel_tf(mashup))   # (1, 128, T)
        return mel, GENRE_TO_IDX[genre]


# ── Build datasets & loaders ──────────────────────────────────────────────────
if KAGGLE_FEATURES.exists():
    print("Kaggle mode: using precomputed .pt features")
    all_files = glob.glob(str(KAGGLE_FEATURES / '**' / '*.pt'), recursive=True)
    random.shuffle(all_files)
    n_val = int(len(all_files) * 0.15)
    train_dataset = PrecomputedFeatureDataset(all_files[n_val:])
    val_dataset   = PrecomputedFeatureDataset(all_files[:n_val])
elif LOCAL_STEMS.exists():
    print("Local mode: on-the-fly generation from stems")
    SAMPLES_PER_GENRE = 20   # small for local; use 50+ on Kaggle
    train_dataset = OnTheFlyDataset(LOCAL_STEMS, GENRES, SAMPLES_PER_GENRE, split='train')
    val_dataset   = OnTheFlyDataset(LOCAL_STEMS, GENRES, max(1, SAMPLES_PER_GENRE // 5), split='val')
else:
    raise RuntimeError("No data found. Set LOCAL_STEMS to your stems directory.")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)},  val samples: {len(val_dataset)}")
print(f"Train batches: {len(train_loader)},    val batches: {len(val_loader)}")


Local mode: on-the-fly generation from stems
Train samples: 200,  val samples: 40
Train batches: 7,    val batches: 2


In [19]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
import numpy as np

# ── Training configuration ─────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

print(f"Device: {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Epochs: {num_epochs}\n")

# ── Training loop ──────────────────────────────────────────────────────────────
for epoch in range(1, num_epochs + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_loss, train_correct, n_train = 0.0, 0, 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * labels.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        n_train       += labels.size(0)

    # --- Validate ---
    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            logits   = model(features)
            val_loss += criterion(logits, labels).item() * labels.size(0)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    n_val     = len(all_labels)
    train_acc = train_correct / n_train
    val_acc   = sum(p == l for p, l in zip(all_preds, all_labels)) / n_val
    val_f1    = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    elapsed   = time.time() - t0

    print(f"Epoch {epoch:2d}/{num_epochs} | "
          f"TrainLoss: {train_loss/n_train:.4f}  TrainAcc: {train_acc:.3f} | "
          f"ValLoss: {val_loss/n_val:.4f}  ValAcc: {val_acc:.3f}  ValF1: {val_f1:.3f} | "
          f"{elapsed:.0f}s")

print("\nTraining complete.")


Device: cpu
Model params: 1,102,666
Epochs: 10

Epoch  1/10 | TrainLoss: 2.1685  TrainAcc: 0.215 | ValLoss: 2.1190  ValAcc: 0.325  ValF1: 0.194 | 133s
Epoch  2/10 | TrainLoss: 1.8461  TrainAcc: 0.465 | ValLoss: 1.9585  ValAcc: 0.325  ValF1: 0.207 | 139s
Epoch  3/10 | TrainLoss: 1.7094  TrainAcc: 0.555 | ValLoss: 1.8277  ValAcc: 0.425  ValF1: 0.315 | 235s
Epoch  4/10 | TrainLoss: 1.6494  TrainAcc: 0.510 | ValLoss: 1.9285  ValAcc: 0.325  ValF1: 0.192 | 251s
Epoch  5/10 | TrainLoss: 1.5093  TrainAcc: 0.600 | ValLoss: 1.8796  ValAcc: 0.375  ValF1: 0.229 | 178s
Epoch  6/10 | TrainLoss: 1.4420  TrainAcc: 0.715 | ValLoss: 1.8235  ValAcc: 0.400  ValF1: 0.240 | 167s
Epoch  7/10 | TrainLoss: 1.3597  TrainAcc: 0.675 | ValLoss: 1.7593  ValAcc: 0.375  ValF1: 0.257 | 166s
Epoch  8/10 | TrainLoss: 1.2558  TrainAcc: 0.720 | ValLoss: 1.7764  ValAcc: 0.350  ValF1: 0.259 | 242s
Epoch  9/10 | TrainLoss: 1.2238  TrainAcc: 0.735 | ValLoss: 1.7683  ValAcc: 0.350  ValF1: 0.254 | 214s
Epoch 10/10 | TrainLoss: 